[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/11_Trajectory_Tracking.ipynb)

# DiveLab

## Notebook 11 — Trajectory Tracking and Ascent Profiles

**Guiding question:** How can a diver follow a desired ascent profile instead of simply holding one depth?

So far we have mostly studied **regulation**:

> keep the system near one equilibrium.

Now we study **tracking**:

> make the system follow a time-varying reference.

This moves DiveLab from stabilization toward mission-level control.

## Learning objectives

By the end of this lab, you will be able to:

- distinguish regulation from trajectory tracking;
- define a time-varying depth reference;
- derive a desired vertical velocity from the reference;
- compare step commands with smooth trajectories;
- use feedback on tracking error;
- introduce feedforward action;
- impose ascent-rate and actuator limits;
- evaluate tracking error quantitatively.

# 1. Regulation vs tracking

In regulation, the target is constant:

$$
z_r(t)=z_e.
$$

In tracking, the target changes with time:

$$
z_r=z_r(t).
$$

For example, an ascent profile might move gradually from:

$$
30\ \text{m}
$$

toward:

$$
5\ \text{m}.
$$

The controller must not only be stable.

It must follow the reference.

## Why not command the final depth instantly?

A naive command could be:

> go from 30 m directly to 5 m.

Mathematically that is a step reference.

But a physical diver cannot jump in depth.

A large step can demand:

- large acceleration;
- large control action;
- actuator saturation;
- excessive vertical velocity.

So trajectory design matters.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 2. Reuse the nonlinear diver model

In [ ]:
rho = 1025.0
g = 9.80665
P0 = 101325.0

mass = 90.0

Cd = 0.9
A_drag = 0.7

Vs_e = 0.005
z_equilibrium_reference = 20.0

def pressure_at_depth(z):
    return P0 + rho * g * z

def gas_volume_at_depth(z, surface_gas_volume):
    return surface_gas_volume * P0 / pressure_at_depth(z)

Vg_eq = gas_volume_at_depth(z_equilibrium_reference, Vs_e)
fixed_volume = mass / rho - Vg_eq

def buoyant_force(z, Vs):
    Vg = gas_volume_at_depth(z, Vs)
    return rho * g * (fixed_volume + Vg)

def drag_force(v):
    return 0.5 * rho * Cd * A_drag * v * abs(v)

def nonlinear_acceleration(z, v, Vs, disturbance_force=0.0):
    return (
        buoyant_force(z, Vs)
        - mass * g
        - drag_force(v)
        + disturbance_force
    ) / mass

# 3. Sign convention reminder

We use:

- depth $z>0$ downward;
- vertical velocity $v>0$ upward.

Therefore:

$$
\dot z=-v.
$$

If the reference depth decreases during ascent:

$$
\dot z_r<0,
$$

then the corresponding desired upward velocity is:

$$
v_r=-\dot z_r>0.
$$

# 4. A simple ascent reference

Start at:

$$
z_0=30\ \text{m}
$$

and target:

$$
z_f=5\ \text{m}.
$$

Suppose we want a constant commanded upward velocity:

$$
v_r=0.15\ \text{m/s}.
$$

Then:

$$
\dot z_r=-0.15.
$$

In [ ]:
z_start = 30.0
z_final = 5.0

v_ref_const = 0.15  # m/s upward

In [ ]:
def constant_rate_reference(t):
    z = z_start - v_ref_const * t
    return max(z, z_final)

## How long does the nominal ascent take?

Ignoring the final hold:

$$
T=
\frac{z_0-z_f}{v_r}.
$$

In [ ]:
T_nominal = (z_start - z_final) / v_ref_const
print(f"Nominal ascent time: {T_nominal:.1f} s")

# 5. Plot the reference

In [ ]:
t_ref = np.linspace(0, T_nominal + 30, 800)
z_ref = np.array([constant_rate_reference(t) for t in t_ref])

plt.plot(t_ref, z_ref)

plt.xlabel("Time [s]")
plt.ylabel("Reference depth [m]")
plt.title("Constant-rate ascent reference")
plt.grid(True)
plt.show()

This reference has a corner when the final depth is reached.

The desired velocity changes suddenly from:

$$
0.15\ \text{m/s}
$$

to:

$$
0.
$$

Later we will replace this with a smoother reference.

# 6. Tracking error

Define depth tracking error:

$$
e_z=z-z_r.
$$

And velocity tracking error:

$$
e_v=v-v_r.
$$

A simple tracking controller can use both:

$$
u=
K_z e_z
-
K_v e_v
-
K_V(V_s-V_{s,r}).
$$

## Interpretation of the signs

If the diver is deeper than the reference:

$$
e_z>0,
$$

the controller should add buoyancy.

If the diver is rising faster than desired:

$$
e_v>0,
$$

the controller should reduce buoyancy.

This is again negative feedback, now around a moving reference.

In [ ]:
Kz = 8e-5
Kv = 9e-4
Kv_s = 0.20

u_max = 0.00035

# 7. Reference velocity

For the constant-rate profile:

$$
v_r=
0.15\ \text{m/s}
$$

until the final depth is reached, then:

$$
v_r=0.
$$

In [ ]:
def constant_rate_reference_with_velocity(t):
    z = z_start - v_ref_const * t

    if z > z_final:
        return z, v_ref_const

    return z_final, 0.0

# 8. First tracking controller

In [ ]:
def tracking_controller(z, v, Vs, t):
    z_r, v_r = constant_rate_reference_with_velocity(t)

    e_z = z - z_r
    e_v = v - v_r

    return (
        Kz * e_z
        - Kv * e_v
        - Kv_s * (Vs - Vs_e)
    )

# 9. Nonlinear tracking simulator

In [ ]:
def simulate_tracking(
    z0=z_start,
    v0=0.0,
    Vs0=Vs_e,
    duration=T_nominal + 40.0,
    dt=0.01,
    controller_fn=tracking_controller,
    disturbance_fn=None,
    u_limit=u_max,
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    z = np.zeros(n)
    v = np.zeros(n)
    Vs = np.zeros(n)
    u = np.zeros(n)

    z_r_hist = np.zeros(n)
    v_r_hist = np.zeros(n)

    z[0] = z0
    v[0] = v0
    Vs[0] = Vs0

    for k in range(n - 1):
        z_r, v_r = constant_rate_reference_with_velocity(t[k])
        z_r_hist[k] = z_r
        v_r_hist[k] = v_r

        uk = controller_fn(z[k], v[k], Vs[k], t[k])
        uk = np.clip(uk, -u_limit, u_limit)
        u[k] = uk

        disturbance = 0.0 if disturbance_fn is None else disturbance_fn(t[k])

        a = nonlinear_acceleration(
            z[k], v[k], Vs[k], disturbance
        )

        v[k + 1] = v[k] + a * dt
        z[k + 1] = max(z[k] - v[k + 1] * dt, 0.0)
        Vs[k + 1] = max(Vs[k] + uk * dt, 0.0)

    z_r_hist[-1], v_r_hist[-1] = constant_rate_reference_with_velocity(t[-1])
    u[-1] = u[-2]

    return t, z, v, Vs, u, z_r_hist, v_r_hist

In [ ]:
t, z, v, Vs, u, z_r, v_r = simulate_tracking()

# 10. Depth tracking

In [ ]:
plt.plot(t, z_r, label="Reference depth")
plt.plot(t, z, label="Actual depth")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Tracking a constant-rate ascent profile")
plt.grid(True)
plt.legend()
plt.show()

The controller is no longer trying to drive depth immediately to 5 m.

Instead, it tries to follow the entire path:

$$
z_r(t).
$$

# 11. Velocity tracking

In [ ]:
plt.plot(t, v_r, label="Reference upward velocity")
plt.plot(t, v, label="Actual upward velocity")

plt.xlabel("Time [s]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Vertical-velocity tracking")
plt.grid(True)
plt.legend()
plt.show()

Tracking velocity explicitly helps prevent the controller from producing unnecessarily aggressive motion while chasing the depth reference.

# 12. Tracking error

In [ ]:
e_z = z - z_r
e_v = v - v_r

plt.plot(t, e_z, label="Depth error")
plt.plot(t, e_v, label="Velocity error")

plt.axhline(0, linestyle="--")
plt.xlabel("Time [s]")
plt.ylabel("Tracking error")
plt.title("Tracking errors")
plt.grid(True)
plt.legend()
plt.show()

# 13. Quantify performance

Useful tracking metrics include:

### Root mean squared error

$$
RMSE_z=
\sqrt{
\frac{1}{N}
\sum_k e_{z,k}^2
}.
$$

### Maximum absolute error

$$
e_{\max}=
\max_k |e_{z,k}|.
$$

In [ ]:
rmse_z = np.sqrt(np.mean(e_z**2))
max_error_z = np.max(np.abs(e_z))

print(f"Depth RMSE: {rmse_z:.3f} m")
print(f"Maximum absolute depth error: {max_error_z:.3f} m")

# 14. Control effort

In [ ]:
plt.plot(t, u)
plt.axhline(u_max, linestyle="--")
plt.axhline(-u_max, linestyle="--")

plt.xlabel("Time [s]")
plt.ylabel("Control input u [m³/s]")
plt.title("Tracking control effort")
plt.grid(True)
plt.show()

Good tracking alone is not enough.

A controller that tracks perfectly by saturating the actuator continuously may be impractical.

So trajectory tracking always involves tradeoffs between:

- tracking accuracy;
- smoothness;
- actuator effort;
- physical limits.

# 15. Why feedforward helps

Feedback reacts after tracking error appears.

But the desired trajectory already tells us what motion is intended.

That information can be used in advance.

This is **feedforward**.

## Feedback

Feedback says:

> "I am off the desired path. Correct the error."

## Feedforward

Feedforward says:

> "I know the path is about to require upward motion, so prepare the plant before large error develops."

The two are complementary.

# 16. Simple feedforward idea

For a desired vertical velocity $v_r$, hydrodynamic drag is approximately:

$$
F_D(v_r)
=
\frac12\rho C_DA\,v_r|v_r|.
$$

A feedforward term can try to create enough additional buoyancy to balance that drag during steady ascent.

At constant velocity:

$$
\dot v\approx0,
$$

so approximately:

$$
F_B-mg-F_D(v_r)\approx0.
$$

This lets us estimate the gas-volume adjustment needed to sustain the desired ascent rate.

It will not be exact, because the plant is nonlinear and the gas state evolves, but it provides a useful anticipatory input.

In [ ]:
def desired_surface_gas_for_steady_velocity(z_r, v_r):
    desired_buoyant_force = mass * g + drag_force(v_r)

    required_total_volume = desired_buoyant_force / (rho * g)
    required_gas_volume_at_depth = required_total_volume - fixed_volume

    required_surface_volume = (
        required_gas_volume_at_depth
        * pressure_at_depth(z_r) / P0
    )

    return max(required_surface_volume, 0.0)

# 17. Reference gas state

In [ ]:
Vs_ref_profile = np.array([
    desired_surface_gas_for_steady_velocity(zr, vr)
    for zr, vr in zip(z_r, v_r)
])

plt.plot(t, Vs_ref_profile * 1000)

plt.xlabel("Time [s]")
plt.ylabel("Reference surface-equivalent gas [L]")
plt.title("Feedforward gas-state reference")
plt.grid(True)
plt.show()

This profile estimates how the BCD gas state should vary to support the planned motion.

Now feedback only needs to correct model error and disturbances.

# 18. Feedforward + feedback controller

In [ ]:
def ff_tracking_controller(z, v, Vs, t):
    z_r, v_r = constant_rate_reference_with_velocity(t)

    Vs_r = desired_surface_gas_for_steady_velocity(z_r, v_r)

    e_z = z - z_r
    e_v = v - v_r
    e_Vs = Vs - Vs_r

    return (
        Kz * e_z
        - Kv * e_v
        - Kv_s * e_Vs
    )

In [ ]:
t_ff, z_ff, v_ff, Vs_ff, u_ff, z_r_ff, v_r_ff = simulate_tracking(
    controller_fn=ff_tracking_controller
)

# 19. Compare feedback-only and feedforward + feedback

In [ ]:
plt.plot(t, z_r, label="Reference")
plt.plot(t, z, label="Feedback only")
plt.plot(t_ff, z_ff, label="Feedforward + feedback")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Tracking architecture comparison")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
e_ff = z_ff - z_r_ff

print(f"Feedback-only RMSE: {np.sqrt(np.mean(e_z**2)):.3f} m")
print(f"FF + FB RMSE:       {np.sqrt(np.mean(e_ff**2)):.3f} m")

Feedforward does not replace feedback.

If the model is imperfect, pure feedforward will be wrong.

A strong architecture is:

> feedforward for the planned motion + feedback for correction.

# 20. Step reference vs trajectory reference

Let's compare a sudden command to 5 m with the gradual ascent profile.

In [ ]:
def step_reference_with_velocity(t):
    return z_final, 0.0

def step_controller(z, v, Vs, t):
    z_r_step, v_r_step = step_reference_with_velocity(t)

    return (
        Kz * (z - z_r_step)
        - Kv * (v - v_r_step)
        - Kv_s * (Vs - Vs_e)
    )

In [ ]:
def simulate_step_reference():
    n = int((T_nominal + 40.0) / 0.01) + 1
    t_s = np.linspace(0, T_nominal + 40.0, n)

    z_s = np.zeros(n)
    v_s = np.zeros(n)
    Vs_s = np.zeros(n)
    u_s = np.zeros(n)

    z_s[0] = z_start
    Vs_s[0] = Vs_e

    for k in range(n - 1):
        uk = np.clip(
            step_controller(z_s[k], v_s[k], Vs_s[k], t_s[k]),
            -u_max, u_max
        )

        a = nonlinear_acceleration(z_s[k], v_s[k], Vs_s[k])

        v_s[k + 1] = v_s[k] + a * 0.01
        z_s[k + 1] = max(z_s[k] - v_s[k + 1] * 0.01, 0.0)
        Vs_s[k + 1] = max(Vs_s[k] + uk * 0.01, 0.0)

        u_s[k] = uk

    u_s[-1] = u_s[-2]
    return t_s, z_s, v_s, Vs_s, u_s

t_step, z_step, v_step, Vs_step, u_step = simulate_step_reference()

In [ ]:
plt.plot(t_step, z_step, label="Step command")
plt.plot(t, z, label="Trajectory command")
plt.axhline(z_final, linestyle="--", label="Final depth")

plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Step regulation vs trajectory tracking")
plt.grid(True)
plt.legend()
plt.show()

A trajectory reference can reduce the tendency to demand excessive transient motion.

This is a major systems-engineering lesson:

> good control begins with a physically sensible reference.

# 21. Smooth reference instead of a corner

The constant-rate profile has abrupt changes in velocity at:

- the start;
- the final depth.

We can make the motion smoother with a cubic interpolation.

Let:

$$
s=\frac{t}{T},
$$

and use:

$$
h(s)=3s^2-2s^3.
$$

Then:

$$
z_r(t)=z_0+(z_f-z_0)h(s).
$$

This gives zero reference velocity at both endpoints.

In [ ]:
T_smooth = 180.0

def smooth_reference(t):
    if t <= 0:
        return z_start, 0.0

    if t >= T_smooth:
        return z_final, 0.0

    s = t / T_smooth

    h = 3*s**2 - 2*s**3
    dh_ds = 6*s - 6*s**2

    z_r = z_start + (z_final - z_start) * h
    dzdt = (z_final - z_start) * dh_ds / T_smooth

    v_r = -dzdt

    return z_r, v_r

In [ ]:
t_smooth = np.linspace(0, T_smooth + 30, 800)

z_smooth_ref = []
v_smooth_ref = []

for tt in t_smooth:
    zr, vr = smooth_reference(tt)
    z_smooth_ref.append(zr)
    v_smooth_ref.append(vr)

z_smooth_ref = np.array(z_smooth_ref)
v_smooth_ref = np.array(v_smooth_ref)

plt.plot(t_smooth, z_smooth_ref)

plt.xlabel("Time [s]")
plt.ylabel("Reference depth [m]")
plt.title("Smooth ascent reference")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(t_smooth, v_smooth_ref)

plt.xlabel("Time [s]")
plt.ylabel("Reference upward velocity [m/s]")
plt.title("Smooth reference velocity")
plt.grid(True)
plt.show()

The smooth reference avoids instantaneous velocity changes.

This is easier for both the plant and the controller.

# 22. Reference shaping

The process of designing a physically reasonable command is called **reference shaping**.

Instead of asking the controller to solve an impossible transient problem, we give it a reference that already respects desired motion characteristics.

Possible constraints include:

- maximum vertical velocity;
- maximum acceleration;
- smooth start and stop;
- actuator authority.

# 23. Velocity limits

Suppose a trajectory specification imposes:

$$
v_r(t)\le v_{\max}.
$$

We can check whether a proposed reference satisfies that limit before simulating the plant.

In [ ]:
v_max = 0.20

print(f"Maximum smooth reference velocity: {np.max(v_smooth_ref):.3f} m/s")
print(f"Chosen velocity limit:             {v_max:.3f} m/s")
print("Reference respects limit:", np.max(v_smooth_ref) <= v_max)

This illustrates an important separation:

### Planner

Generates a feasible trajectory.

### Controller

Makes the physical plant follow it.

This is the beginning of a hierarchical control architecture.

# 24. Planning vs control

A mission-level system may contain:

```text
MISSION GOAL
     |
     v
TRAJECTORY PLANNER
     |
     v
REFERENCE z_r(t), v_r(t)
     |
     v
TRACKING CONTROLLER
     |
     v
ACTUATOR
     |
     v
DIVER
```

The planner decides **what motion should occur**.

The controller decides **how to make it occur**.

# 25. Add a disturbance during tracking

In [ ]:
def tracking_disturbance(t):
    if 80 <= t <= 85:
        return 20.0
    return 0.0

In [ ]:
t_dist, z_dist, v_dist, Vs_dist, u_dist, zrd, vrd = simulate_tracking(
    disturbance_fn=tracking_disturbance
)

In [ ]:
plt.plot(t_dist, zrd, label="Reference")
plt.plot(t_dist, z_dist, label="Actual depth")

plt.axvspan(80, 85, alpha=0.15)
plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Trajectory tracking with disturbance")
plt.grid(True)
plt.legend()
plt.show()

Tracking control must do two things at once:

1. follow the planned trajectory;
2. reject disturbances.

This is harder than simple depth holding.

# 26. Path error vs final error

A controller could reach the final target correctly while tracking the intermediate trajectory poorly.

Therefore we care about the **entire path**, not just the final state.

This is why trajectory metrics such as RMSE are useful.

In [ ]:
tracking_rmse = np.sqrt(np.mean((z_dist - zrd)**2))
final_error = z_dist[-1] - zrd[-1]

print(f"Tracking RMSE: {tracking_rmse:.3f} m")
print(f"Final depth error: {final_error:.3f} m")

A small final error does not guarantee good trajectory tracking.

# 27. Systems-theory interpretation

Trajectory tracking changes the control objective.

Instead of stabilizing:

$$
x=0,
$$

we define an error state:

$$
e=x-x_r(t).
$$

The control problem becomes:

> stabilize the tracking error around zero.

This is a powerful idea because tracking can often be transformed into regulation of an error system.

# 28. Where estimation fits

In a realistic system, the controller would not use the true state.

It would use:

$$
\hat x
$$

from the Kalman filter developed in Notebook 08 and Notebook 09.

So the realistic tracking loop becomes:

```text
trajectory planner
       |
       v
reference
       |
       v
controller <----- state estimate
                     ^
                     |
               Kalman filter
                     ^
                     |
                 depth sensor
```

This combines planning, estimation and control.

# Exercises

### 1. Change ascent rate

Try:

```python
v_ref_const = 0.05
v_ref_const = 0.10
v_ref_const = 0.20
```

Compare:

- tracking error;
- velocity;
- saturation.

### 2. Change final depth

Try:

```python
z_final = 10.0
```

and:

```python
z_final = 3.0
```

How does shallower operation affect the nonlinear plant?

### 3. Remove velocity feedback

Set:

```python
Kv = 0
```

How does the tracking response change?

### 4. Compare step and smooth references

Evaluate:

- maximum velocity;
- maximum control effort;
- tracking RMSE.

Which reference is easier to follow?

### 5. Disturbance rejection during ascent

Change the disturbance:

- magnitude;
- duration;
- timing.

Does the controller recover while still following the trajectory?

# Challenge — estimated-state trajectory tracking

Combine Notebook 09 with this notebook.

Use:

$$
u=-K(\hat x-x_r)
$$

where $\hat x$ comes from a Kalman filter and $x_r(t)$ is the tracking reference.

Add:

- measurement noise;
- process disturbances;
- actuator saturation.

Compare perfect-state and estimated-state tracking.

In [ ]:
# Your code here

# Summary

In this notebook we moved from regulation to trajectory tracking.

We learned that:

- regulation holds one operating point;
- tracking follows a time-varying reference;
- depth and velocity references can be used together;
- trajectory tracking can be written as error regulation;
- feedforward anticipates planned motion;
- feedback corrects model error and disturbances;
- smooth reference shaping reduces aggressive transients;
- velocity and actuator constraints belong in the planning problem;
- trajectory planning and feedback control form a hierarchy;
- realistic tracking will eventually require state estimation as well.

### Core insight

> **A good controller should not merely reach the destination. It should follow a physically meaningful path to get there.**

### Next

Notebook 12 can introduce **constraints and predictive control**:

> Can the controller predict future motion and choose actions that respect depth, velocity and actuator constraints?

That leads naturally to **Model Predictive Control (MPC)**.